# LangChain Runnables -- Extended Deep-Dive
> Picks up where the base notebook left off. Each section adds genuinely new territory.

**Sections:**
1. `with_structured_output()` -- typed output without parsing
2. Tool Calling -- binding functions to LLMs
3. Custom Runnable Subclass -- stateful components
4. `RunnablePassthrough.assign()` -- deep enrichment
5. LCEL Agent Loop -- manual ReAct
6. Token Counting and Cost Estimation
7. Summarization Memory -- long conversations
8. Rate Limiting and Circuit Breaker
9. FastAPI Integration -- streaming endpoint
10. Testing LCEL Chains -- mocking and evals


## Setup
Run this cell before any other.

In [ ]:
import os, json, re, time, asyncio, hashlib, threading
from datetime import datetime
from typing import Any, Optional
from dotenv import load_dotenv
load_dotenv()

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda, RunnableParallel, RunnableBranch,
    RunnablePassthrough, RunnableConfig, ConfigurableField, Runnable
)
from langchain_core.tracers.schemas import Run
from langchain_core.tools import tool
from pydantic import BaseModel, Field

model  = ChatGroq(model='llama-3.3-70b-versatile')
parser = StrOutputParser()
print('Setup complete')


---
## 1. `with_structured_output()` -- The Modern Typed Output

### Why this is better than PydanticOutputParser

The old approach *prompts* the model to return JSON, then parses the text.
If the model adds a sentence before the JSON, the parser crashes.

`with_structured_output()` uses the model's **native function-calling mechanism** --
the API forces the model to fill a schema. It cannot fail due to model formatting.

```
PydanticOutputParser:   prompt says 'return JSON' -> model text -> you parse -> might crash
with_structured_output: API sends schema -> model fills it -> API returns object -> safe
```

### Critical: the output type changes

After `with_structured_output`, the model returns your Pydantic object, not an `AIMessage`.
Do **not** add `StrOutputParser` after it -- there's no message to unwrap.

```python
# WRONG
chain = prompt | model.with_structured_output(MySchema) | StrOutputParser()  # crashes

# CORRECT -- the object IS your final output
chain = prompt | model.with_structured_output(MySchema)
result = chain.invoke(input)
result.field_name   # direct attribute access
```


In [ ]:
# 1a. Simple schema
class SentimentAnalysis(BaseModel):
    sentiment:  str   = Field(description='positive, negative, or neutral')
    confidence: float = Field(description='0.0 to 1.0')
    reason:     str   = Field(description='one sentence why')

sentiment_chain = (
    ChatPromptTemplate.from_template('Analyze sentiment of: {text}')
    | model.with_structured_output(SentimentAnalysis)
)

r = sentiment_chain.invoke({'text': 'This library is confusing but powerful.'})
print(f'type:       {type(r).__name__}')  # SentimentAnalysis -- not str!
print(f'sentiment:  {r.sentiment}')
print(f'confidence: {r.confidence}')
print(f'reason:     {r.reason}')
print()

# 1b. Nested schema -- extract complex structure from plain text
class Entity(BaseModel):
    name: str = Field(description='entity name')
    type: str = Field(description='person, place, org, or concept')

class NERResult(BaseModel):
    entities:     list[Entity]
    main_topic:   str
    is_technical: bool

ner_chain = (
    ChatPromptTemplate.from_template('Extract structured info from:\n\n{text}')
    | model.with_structured_output(NERResult)
)

text = ('Geoffrey Hinton left Google to speak about AI risks. '
        'His work at the University of Toronto changed ML forever.')

ner = ner_chain.invoke({'text': text})
print(f'main_topic:   {ner.main_topic}')
print(f'is_technical: {ner.is_technical}')
for e in ner.entities:
    print(f'  [{e.type}] {e.name}')
print()

# 1c. MODIFICATION: pipe structured output into a formatter
# The original files never showed this -- structured output IS composable with |
def render_ner(result: NERResult) -> str:
    by_type = {}
    for e in result.entities:
        by_type.setdefault(e.type, []).append(e.name)
    lines = [f'Topic: {result.main_topic}', '']
    for t, names in sorted(by_type.items()):
        lines.append(f'  {t}: {', '.join(names)}')
    return '\n'.join(lines)

ner_report_chain = (
    ChatPromptTemplate.from_template('Extract entities from: {text}')
    | model.with_structured_output(NERResult)
    | RunnableLambda(render_ner)   # structured object -> formatted string
)
print('-- NER piped into formatter --')
print(ner_report_chain.invoke({'text': text}))


---
## 2. Tool Calling -- Binding Functions to LLMs

### What tool calling actually is

```
Model decides:    WHICH tool to call and with WHAT arguments
Model does NOT:   Execute the tool

Your code does:   Execute the tool, get result
Your code does NOT: Decide when to call it
```

The model returns `AIMessage` with a `tool_calls` field. Your code reads it,
runs the function, wraps the result in `ToolMessage`, sends it back.

### The @tool decorator

```python
@tool
def my_fn(arg: str) -> str:
    'Docstring becomes the tool description -- model reads this to decide when to call it.'
    return f'result: {arg}'
```

Decorator reads: signature -> JSON schema; docstring -> tool description.

### After bind_tools, always check tool_calls first

```python
response = model.bind_tools([fn]).invoke(messages)
if response.tool_calls:
    # model wants a tool -- response.content might be empty
else:
    # model answered directly -- use response.content
```


In [ ]:
@tool
def calculate(expression: str) -> float:
    'Evaluate a Python math expression. Use math module functions like sqrt, pi.'
    import math
    return eval(expression, {'__builtins__': {}}, vars(math))

@tool
def word_stats(text: str) -> dict:
    'Count words, characters, and sentences in a text string.'
    words = text.split()
    sents = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    return {'words': len(words), 'chars': len(text), 'sentences': len(sents)}

tools    = [calculate, word_stats]
tool_map = {t.name: t for t in tools}
model_wt = model.bind_tools(tools)

# Does the model call a tool?
print('-- Does model request a tool? --')
response = model_wt.invoke([HumanMessage(content='What is 17 * 23 + sqrt(144)?')])
print(f'has tool_calls: {bool(response.tool_calls)}')
if response.tool_calls:
    for tc in response.tool_calls:
        print(f'  tool={tc["name"]}, args={tc["args"]}')
print()

# Complete the loop
def run_tool_loop(question: str, verbose=True) -> str:
    # user message -> model response -> (maybe) tool execution -> final model response
    # We loop until tool_calls is empty (model answered)
    msgs = [HumanMessage(content=question)]
    for _ in range(5):              # safety cap: max 5 iterations
        resp = model_wt.invoke(msgs)
        msgs.append(resp)
        if not resp.tool_calls:
            return resp.content     # done
        for tc in resp.tool_calls:
            result = tool_map[tc['name']].invoke(tc['args'])
            if verbose: print(f'  [tool] {tc["name"]}({tc["args"]}) -> {result}')
            msgs.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
    return 'Max iterations reached'

print('-- Full tool loop --')
answer = run_tool_loop('What is 17 * 23 + sqrt(144)?')
print('Final:', answer)


---
## 3. Custom Runnable Subclass -- Stateful, Reusable Components

### When to subclass instead of using Lambda

Use `RunnableLambda` for stateless, simple logic.
Write a **class** when you need:
- State: a cache, a counter, a database connection
- Init parameters that control behavior
- Reusability across projects

### Minimal recipe

```python
class MyRunnable(Runnable):
    def invoke(self, input: Any, config=None) -> Any:
        return your_logic(input)
# batch / stream / ainvoke / | -> inherited for free from Runnable base class
```

### Three practical patterns

**CachingRunnable** -- memoize expensive LLM calls  
**AuditRunnable** -- structured compliance logging  
**TokenGuardRunnable** -- reject inputs exceeding a token budget


In [ ]:
try:
    import tiktoken
except ImportError:
    import subprocess; subprocess.run(['pip','install','tiktoken','-q','--break-system-packages'])
    import tiktoken

def count_tokens(text: str) -> int:
    # cl100k_base is GPT-4's tokenizer -- close approximation for LLaMA too
    enc = tiktoken.get_encoding('cl100k_base')
    return len(enc.encode(text))

# -- CachingRunnable --------------------------------------------------------
class CachingRunnable(Runnable):
    # Wraps any Runnable and caches results by input.
    # Cache key: SHA256 of JSON-serialized input (works for dicts/strings/lists).
    # Use when: same prompts appear repeatedly in batch processing.
    def __init__(self, runnable: Runnable, max_size: int = 200):
        self.runnable = runnable
        self.max_size = max_size
        self._cache = {}
        self.hits = self.misses = 0

    def _key(self, input):
        return hashlib.sha256(json.dumps(input, sort_keys=True, default=str).encode()).hexdigest()[:16]

    def invoke(self, input, config=None):
        k = self._key(input)
        if k in self._cache:
            self.hits += 1
            return self._cache[k]
        self.misses += 1
        result = self.runnable.invoke(input, config)
        if len(self._cache) >= self.max_size:
            del self._cache[next(iter(self._cache))]   # evict oldest (LRU)
        self._cache[k] = result
        return result

    def stats(self):
        total = self.hits + self.misses
        return {'hits': self.hits, 'misses': self.misses,
                'hit_rate': round(self.hits/total, 2) if total else 0}

# -- TokenGuardRunnable -----------------------------------------------------
class TokenGuardRunnable(Runnable):
    # Rejects inputs exceeding a token budget BEFORE the API call.
    # Prevents: expensive accidental calls, context-overflow errors.
    def __init__(self, runnable: Runnable, max_tokens: int = 2000):
        self.runnable   = runnable
        self.max_tokens = max_tokens

    def invoke(self, input, config=None):
        n = count_tokens(json.dumps(input, default=str))
        if n > self.max_tokens:
            raise ValueError(f'Input too long: {n} tokens (max {self.max_tokens})')
        return self.runnable.invoke(input, config)

# -- Demo -------------------------------------------------------------------
base_chain = ChatPromptTemplate.from_template('Capital of {country}? One word.') | model | parser
cached     = CachingRunnable(base_chain)

print('-- CachingRunnable: 5 calls, 3 unique inputs --')
for c in ['France','Japan','France','Japan','Germany']:
    r = cached.invoke({'country': c})
    print(f'  {c:10s}: {r.strip()}')
print('Stats:', cached.stats(), '<-- only 3 API calls')
print()

guarded = TokenGuardRunnable(base_chain, max_tokens=50)
print('-- TokenGuardRunnable --')
print('Short ok:', guarded.invoke({'country': 'Brazil'}).strip())
try:
    guarded.invoke({'country': 'word ' * 100})
except ValueError as e:
    print('Long blocked:', str(e)[:70])
print()

# Custom Runnables compose with | just like built-in ones
safe_cached = TokenGuardRunnable(CachingRunnable(base_chain))
print('-- Composed: guard | cache | chain --')
for c in ['Canada', 'India', 'Canada']:
    print(f'  {c}: {safe_cached.invoke({"country": c}).strip()}')


---
## 4. `RunnablePassthrough.assign()` -- Deep Enrichment Patterns

### assign vs RunnableParallel -- a subtle but important difference

```python
# RunnableParallel REPLACES the flowing dict
RunnableParallel({'a': chain_a}).invoke({'x': 1})
# -> {'a': result}   <-- original 'x' is GONE

# assign MERGES computed keys INTO the existing dict
RunnablePassthrough.assign(a=chain_a).invoke({'x': 1})
# -> {'x': 1, 'a': result}   <-- 'x' is preserved
```

Use `assign` when downstream stages need both original keys AND computed ones.

### Chained assigns -- progressive enrichment

```python
pipeline = (
    RunnablePassthrough.assign(claim=claim_chain)
    # {'topic': X} -> {'topic': X, 'claim': '...'}

    | RunnablePassthrough.assign(evidence=ev, counter=ctr)  # parallel
    # -> {'topic', 'claim', 'evidence', 'counter'}

    | RunnablePassthrough.assign(verdict=verdict_chain)
    # -> all 5 keys available
)
```

### Key insight: assign once, use in multiple downstream chains

If two later chains both need a `summary`, compute it once with `assign`.
Both see `d['summary']` without a second API call.


In [ ]:
claim_p    = ChatPromptTemplate.from_template('Make one debatable claim about {topic}. One sentence.')
evidence_p = ChatPromptTemplate.from_messages([
    ('system','Give 2 supporting evidence points.'),
    ('user','Topic:{topic}\nClaim:{claim}'),
])
counter_p  = ChatPromptTemplate.from_messages([
    ('system','Give 2 counter-arguments.'),
    ('user','Topic:{topic}\nClaim:{claim}'),
])
verdict_p  = ChatPromptTemplate.from_messages([
    ('system','Impartial judge. Score 1-10 then one sentence.'),
    ('user','Claim:{claim}\nEvidence:{evidence}\nCounters:{counter}'),
])

claim_chain    = claim_p    | model | parser
evidence_chain = evidence_p | model | parser
counter_chain  = counter_p  | model | parser
verdict_chain  = verdict_p  | model | parser

research_pipeline = (
    RunnablePassthrough.assign(claim=claim_chain)
    | RunnablePassthrough.assign(
        evidence=evidence_chain,   # concurrent
        counter=counter_chain,     # same input, runs in parallel
    )
    | RunnablePassthrough.assign(verdict=verdict_chain)
)

print('-- Progressive enrichment pipeline --')
r = research_pipeline.invoke({'topic': 'AI will replace most software engineers'})
print(f'CLAIM:    {r["claim"]}')
print(f'EVIDENCE: {r["evidence"][:80]}...')
print(f'COUNTER:  {r["counter"][:80]}...')
print(f'VERDICT:  {r["verdict"]}')
print()

# Assign once, use twice pattern
summarize_p = ChatPromptTemplate.from_template('Summarize in one sentence: {text}')
expand_p    = ChatPromptTemplate.from_messages([
    ('system','Generate 3 follow-up research questions.'),
    ('user','Summary: {summary}'),
])
tweet_p = ChatPromptTemplate.from_messages([
    ('system','Convert to tweet under 280 chars with hashtags.'),
    ('user','Summary: {summary}'),
])

summarize_chain = summarize_p | model | parser
expand_chain    = expand_p    | model | parser
tweet_chain     = tweet_p     | model | parser

# summary computed ONCE, then available to both expand and tweet chains
smart_pipeline = (
    RunnablePassthrough.assign(summary=summarize_chain)
    | RunnablePassthrough.assign(
        questions=expand_chain,
        tweet=tweet_chain,
    )
)

print('-- assign once, use in 2 chains (no repeated API call) --')
text = 'Transformers replaced RNNs by using self-attention for parallel sequence processing.'
r2 = smart_pipeline.invoke({'text': text})
print(f'SUMMARY:   {r2["summary"]}')
print(f'TWEET:     {r2["tweet"]}')
print(f'QUESTIONS: {r2["questions"][:100]}...')


---
## 5. LCEL Agent Loop -- Manual ReAct Without AgentExecutor

### What an agent is (stripped down)

```python
while True:
    response = model.invoke(conversation)
    if not response.tool_calls:
        return response.content    # done
    execute_tools_and_append_results()
```

`AgentExecutor` wraps this. Building it manually gives you full control:
custom stopping conditions, step-by-step logging, intermediate streaming,
and the agent becomes a composable `Runnable` in a larger pipeline.

### ReAct (Reason + Act) message types

```
AIMessage (no tool_calls) -> Final Answer  -- exit loop
AIMessage (with tool_calls) -> Action      -- execute tools
ToolMessage                -> Observation  -- tool result back to model
```

### AgentExecutor vs raw loop

| AgentExecutor | Raw LCEL Loop |
|---|---|
| Opaque internals | You see every step |
| Fixed loop behavior | Modify any iteration |
| Hard to add custom stopping | Easy |
| Less composable | Works with | operator |


In [ ]:
@tool
def search_fact(query: str) -> str:
    'Search for a factual answer about any topic.'
    facts = {
        'python':    'Python was created by Guido van Rossum, released in 1991.',
        'langchain': 'LangChain is an LLM framework released in 2022.',
        'llm':       'LLMs are transformer-based models trained on large text corpora.',
    }
    key = next((k for k in facts if k in query.lower()), None)
    return facts[key] if key else f'General info about: {query}'

@tool
def extract_year(text: str) -> int:
    'Extract the first 4-digit year from a text string. Returns -1 if none found.'
    years = re.findall(r'\\b(19|20)\\d{2}\\b', text)
    return int(years[0]) if years else -1

@tool
def years_elapsed(start_year: int, end_year: int = 2025) -> int:
    'Calculate years elapsed from start_year to end_year.'
    return end_year - start_year

agent_tools    = [search_fact, extract_year, years_elapsed]
agent_tool_map = {t.name: t for t in agent_tools}
agent_model    = model.bind_tools(agent_tools)

AGENT_SYS = ('You are a research assistant with tools. '
             'Use tools to gather facts. When done, answer directly without more tool calls.')

def run_agent(question: str, max_iter=6, verbose=True) -> str:
    msgs = [SystemMessage(content=AGENT_SYS), HumanMessage(content=question)]
    for i in range(max_iter):
        if verbose: print(f'  [iter {i+1}] calling model...')
        response = agent_model.invoke(msgs)
        msgs.append(response)
        if not response.tool_calls:
            if verbose: print(f'  [done after {i+1} iter]')
            return response.content
        for tc in response.tool_calls:
            result = agent_tool_map[tc['name']].invoke(tc['args'])
            if verbose: print(f'  [tool] {tc["name"]}({tc["args"]}) -> {result}')
            msgs.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
    return 'Max iterations reached'

print('-- Manual Agent Loop --')
answer = run_agent('How old is Python in 2025?')
print('\nFINAL:', answer)
print()

# Wrap as a composable Runnable
agent_runnable = RunnableLambda(lambda q: run_agent(q, verbose=False))
print('-- Agent as Runnable (composable with |) --')
result = agent_runnable.invoke('What is LangChain and when was it released?')
print(result[:200])


---
## 6. Token Counting and Cost Estimation

### Why you need this in production

You can't run an LLM app blind to cost. Two uses:
1. **Pre-flight validation**: reject inputs too long before the API call
2. **Post-call accounting**: log tokens used, compute cost per user

### tiktoken

The standard tokenizer library. For non-OpenAI models (Groq, LLaMA),
use `cl100k_base` as a close approximation:

```python
enc = tiktoken.get_encoding('cl100k_base')
n_tokens = len(enc.encode(text))  # ~5-15% off for LLaMA
```

### Real usage from response metadata

After any `model.invoke()`, check `response.usage_metadata`:
```python
meta = response.usage_metadata   # input_tokens, output_tokens, total_tokens
```


In [ ]:
MODEL_PRICING = {
    'llama-3.3-70b-versatile': {'input': 0.59,  'output': 0.79},   # per 1M tokens
    'gpt-4o':                  {'input': 2.50,  'output': 10.0},
    'gpt-4o-mini':             {'input': 0.15,  'output': 0.60},
    'claude-sonnet-4-6':       {'input': 3.00,  'output': 15.0},
}

def estimate_cost(input_tok: int, output_tok: int, model_name: str) -> dict:
    p  = MODEL_PRICING.get(model_name, {'input': 1.0, 'output': 1.0})
    ic = (input_tok  / 1_000_000) * p['input']
    oc = (output_tok / 1_000_000) * p['output']
    return {'input_cost': round(ic,6), 'output_cost': round(oc,6), 'total': round(ic+oc,6)}

# Token counts on different text sizes
texts = {
    'tweet':     'LangChain makes LLM pipelines composable. #AI',
    'paragraph': 'Transformers use self-attention for parallel sequence processing. ' * 3,
    'article':   'Large language models are trained on vast text corpora. ' * 30,
}
print('-- Token counts --')
for label, t in texts.items():
    print(f'  {label:12s}: {count_tokens(t):4d} tokens')
print()

# Cost comparison across models for a typical RAG call
print('-- Cost for 500 input + 200 output tokens --')
for m in MODEL_PRICING:
    c = estimate_cost(500, 200, m)
    print(f'  {m:40s}: ${c["total"]:.5f}')
print()

# Real usage from the API
print('-- Actual usage_metadata from Groq --')
resp = model.invoke([HumanMessage(content='What is 2+2? One word.')])
if hasattr(resp, 'usage_metadata') and resp.usage_metadata:
    meta = resp.usage_metadata
    print(f'  input_tokens:  {meta.get("input_tokens", "N/A")}')
    print(f'  output_tokens: {meta.get("output_tokens", "N/A")}')
    c = estimate_cost(meta.get('input_tokens',0), meta.get('output_tokens',0),
                      'llama-3.3-70b-versatile')
    print(f'  estimated cost: ${c["total"]:.7f} USD')
else:
    print('  (usage_metadata not available on this endpoint)')


---
## 7. Summarization Memory -- Handling Long Conversations

### Why plain history fails

Appending every message forever causes:
- Token count grows linearly -> expensive
- Eventually exceeds the context window

Three strategies:

| Strategy | Trades off |
|---|---|
| Truncation | Loses old context entirely |
| Sliding window (keep last N) | Fine if recent context is enough |
| **Summarization** | Best: preserves meaning, reduces tokens |

### Algorithm

```
When len(history) > max_messages:
    summary = LLM.summarize(oldest messages)
    history = [SystemMessage(summary)] + last_N_messages_verbatim
```

Why keep recent verbatim: pronouns like 'the second option' or 'that example'
only make sense with exact recent wording -- don't summarize those.


In [ ]:
class SummarizingMemory:
    # max_messages: trigger compression above this count
    # keep_recent:  always keep this many messages verbatim (not summarized)
    def __init__(self, max_messages=10, keep_recent=4):
        self.max_messages = max_messages
        self.keep_recent  = keep_recent
        self.messages     = []
        self._n_summaries = 0

    def _summarize(self, msgs: list) -> str:
        formatted = '\n'.join(
            f'{type(m).__name__}: {m.content[:200]}' for m in msgs
        )
        resp = model.invoke([
            SystemMessage(content='Summarize this conversation. Preserve key facts. Be concise.'),
            HumanMessage(content=formatted)
        ])
        return resp.content

    def add(self, human: str, ai: str):
        self.messages.append(HumanMessage(content=human))
        self.messages.append(AIMessage(content=ai))
        if len(self.messages) > self.max_messages:
            self._compress()

    def _compress(self):
        # Only summarize real messages, not prior summaries (to avoid lossy double-summary)
        to_sum = [m for m in self.messages[:-self.keep_recent]
                  if not (isinstance(m, SystemMessage) and '[SUMMARY' in m.content)]
        kept   = self.messages[-self.keep_recent:]
        if not to_sum:
            self.messages = kept; return
        self._n_summaries += 1
        summary = self._summarize(to_sum)
        self.messages = [
            SystemMessage(content=f'[SUMMARY #{self._n_summaries}]\n{summary}')
        ] + kept
        print(f'  [Memory: {len(to_sum)} msgs -> 1 summary, {len(kept)} kept verbatim]')

    def get_history(self): return self.messages
    def stats(self): return {'messages': len(self.messages), 'compressions': self._n_summaries}

conv_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Remember context from earlier.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])
conv_chain = conv_prompt | model | parser

memory = SummarizingMemory(max_messages=6, keep_recent=2)
turns = [
    'My name is Sagar. I do ML research.',
    'I work on RAG systems and VLMs.',
    'I use Qwen2.5-VL and vLLM for deployment.',
    'I am learning LCEL chains in LangChain right now.',
    'What do you know about my background?',
    'What ML tools have I mentioned?',
]

print('-- Summarizing Memory: 6-turn chat --')
for i, q in enumerate(turns, 1):
    resp = conv_chain.invoke({'question': q, 'history': memory.get_history()})
    memory.add(q, resp)
    print(f'\nTurn {i}: {q[:55]}')
    print(f'Answer: {resp[:80]}...')
    print(f'Stats:  {memory.stats()}')


---
## 8. Rate Limiting and Circuit Breaker

### Why with_retry() is not enough

- No burst control: can still send 100 requests in 1 second
- No circuit breaker: keeps retrying a completely down API
- No shared state across multiple chain instances

### Token Bucket Rate Limiter

```
Bucket fills at rate R tokens/second, up to capacity C.
Each API call costs 1 token. If empty, wait.
Advantage: allows short bursts (up to capacity), enforces long-term rate.
```

### Circuit Breaker States

```
CLOSED    -> normal operation
  | (N failures)
OPEN      -> fail immediately, protect the broken API
  | (after timeout)
HALF_OPEN -> test one request
  | success -> CLOSED | failure -> OPEN
```


In [ ]:
from enum import Enum, auto

class TokenBucket:
    # Thread-safe token bucket rate limiter.
    def __init__(self, rate: float, capacity: int = None):
        self.rate     = rate
        self.capacity = capacity or max(1, int(rate * 2))
        self._tokens  = float(self.capacity)
        self._last    = time.monotonic()
        self._lock    = threading.Lock()

    def acquire(self, timeout=30.0):
        deadline = time.monotonic() + timeout
        while True:
            with self._lock:
                now = time.monotonic()
                self._tokens = min(self.capacity,
                                   self._tokens + (now - self._last) * self.rate)
                self._last = now
                if self._tokens >= 1:
                    self._tokens -= 1
                    return
            if time.monotonic() > deadline:
                raise TimeoutError('Rate limit: timed out waiting for token')
            time.sleep(0.05)

class CircuitState(Enum):
    CLOSED = auto(); OPEN = auto(); HALF_OPEN = auto()

class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=10.0):
        self.threshold = failure_threshold
        self.recovery  = recovery_timeout
        self._state    = CircuitState.CLOSED
        self._failures = 0
        self._last_fail = 0
        self._lock     = threading.Lock()

    @property
    def state(self):
        with self._lock:
            if (self._state == CircuitState.OPEN and
                time.monotonic() - self._last_fail > self.recovery):
                self._state = CircuitState.HALF_OPEN
                print('  [Circuit -> HALF_OPEN]')
            return self._state

    def call(self, fn, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            raise ConnectionError('Circuit OPEN -- failing fast')
        try:
            result = fn(*args, **kwargs)
            with self._lock:
                if self._state == CircuitState.HALF_OPEN:
                    self._state = CircuitState.CLOSED
                    self._failures = 0
                    print('  [Circuit -> CLOSED (recovered)]')
            return result
        except Exception:
            with self._lock:
                self._failures += 1
                self._last_fail = time.monotonic()
                if self._failures >= self.threshold:
                    self._state = CircuitState.OPEN
                    print(f'  [Circuit -> OPEN ({self._failures} failures)]')
            raise

class RateLimitedRunnable(Runnable):
    # Combines rate limiting + circuit breaking. Drop in front of any LLM chain.
    def __init__(self, runnable, rps=2.0, failure_threshold=3, recovery_timeout=10.0):
        self.runnable = runnable
        self.bucket   = TokenBucket(rate=rps)
        self.breaker  = CircuitBreaker(failure_threshold, recovery_timeout)

    def invoke(self, input, config=None):
        self.bucket.acquire()     # wait for rate slot
        return self.breaker.call(self.runnable.invoke, input, config)

# Demo
simple_chain    = ChatPromptTemplate.from_template('In 5 words: {topic}') | model | parser
rate_limited    = RateLimitedRunnable(simple_chain, rps=3.0)

print('-- Rate-limited chain --')
for t in ['neural networks', 'attention mechanism', 'gradient descent']:
    r = rate_limited.invoke({'topic': t})
    print(f'  {t:25s}: {r.strip()}')
print()

# Circuit breaker demo
print('-- Circuit breaker --')
breaker = CircuitBreaker(failure_threshold=2, recovery_timeout=1.5)
count   = [0]

def flaky(n):
    count[0] += 1
    if count[0] < 3:
        raise ConnectionError(f'Failure #{count[0]}')
    return f'Success on call #{count[0]}'

for i in range(6):
    try:
        r = breaker.call(flaky, i)
        print(f'  Call {i}: OK -- {r} [{breaker.state.name}]')
    except Exception as e:
        print(f'  Call {i}: FAIL -- {str(e)[:50]} [{breaker.state.name}]')
    if i == 2:
        print('  [waiting for recovery...]'); time.sleep(1.6)


---
## 9. FastAPI Integration -- Streaming Endpoint

### Why FastAPI + LCEL is the natural pairing

FastAPI is built on `asyncio`. LCEL chains have `ainvoke` and `astream`.
They fit together without any adapter:

```python
@app.post('/chat')
async def chat(req: Request):
    result = await chain.ainvoke(req.dict())   # non-blocking
    return {'response': result}
```

### Server-Sent Events (SSE) for streaming

SSE pushes text chunks from server to browser as tokens arrive.
Each chunk is a line starting with `data: ` followed by the content:

```
data: The
data: quick
data: brown
data: [DONE]
```

### Test without a full server -- async in Jupyter

```python
result = await chain.ainvoke(input)
async for chunk in chain.astream(input):
    print(chunk, end='', flush=True)
```


In [ ]:
# Full server code -- save to server.py and run: uvicorn server:app --reload
SERVER_CODE = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import AsyncGenerator, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableConfig, ConfigurableField
from dotenv import load_dotenv
load_dotenv()

app = FastAPI(title="LCEL API")
model = ChatGroq(model="llama-3.3-70b-versatile").configurable_fields(
    max_tokens=ConfigurableField(id="max_tokens"),
    temperature=ConfigurableField(id="temperature"),
)
chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}"),
    ])
    | model | StrOutputParser()
)

class ChatRequest(BaseModel):
    question:    str
    max_tokens:  Optional[int]   = 500
    temperature: Optional[float] = 0.7

@app.post("/chat")
async def chat(req: ChatRequest):
    # Standard: wait for full response
    config = RunnableConfig(configurable={
        "max_tokens": req.max_tokens, "temperature": req.temperature
    })
    result = await chain.ainvoke({"question": req.question}, config=config)
    return {"response": result}

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    # Streaming: tokens arrive as SSE events
    config = RunnableConfig(configurable={
        "max_tokens": req.max_tokens, "temperature": req.temperature
    })
    async def generator() -> AsyncGenerator[str, None]:
        async for chunk in chain.astream({"question": req.question}, config=config):
            if chunk:
                yield f"data: {chunk}\\n\\n"  # SSE format
        yield "data: [DONE]\\n\\n"
    return StreamingResponse(generator(), media_type="text/event-stream")

@app.get("/health")
async def health():
    return {"status": "ok"}
'''

print('-- FastAPI server code (save to server.py) --')
print(SERVER_CODE[:600] + '\n...(truncated, full code above)')
print()
print('-- Test commands --')
print('curl -X POST http://localhost:8000/chat -H "Content-Type: application/json"')
print('     -d \'{\'question\': \'What is attention?\', \'max_tokens\': 100}\'')
print()

# Test async locally (no full server needed)
print('-- Async test in Jupyter --')

async def test_async():
    p = ChatPromptTemplate.from_template('In 2 sentences, explain {topic}.')
    async_chain = p | model | parser
    # ainvoke: wait for full result
    result = await async_chain.ainvoke({'topic': 'self-attention'})
    print('ainvoke:', result[:80])
    # astream: tokens arrive one by one
    print('astream: ', end='')
    async for chunk in async_chain.astream({'topic': 'LCEL'}):
        print(chunk, end='', flush=True)
    print()

asyncio.run(test_async())


---
## 10. Testing LCEL Chains -- Mocking and Structural Assertions

### Why testing LLM chains is hard

| Problem | Solution |
|---|---|
| Non-determinism: same prompt, different output | Assert structure, not exact content |
| Cost: every test hits a paid API | Mock the model for unit tests |
| Latency: 1-3 seconds per call | Cache in CI; mock in unit tests |

### MockChatModel

Replace `ChatGroq` with a `Runnable` that returns a fixed `AIMessage`.
This makes tests deterministic, instant, and free.

### Structural assertions

```python
# Bad: brittle exact match
assert result == 'Paris is the capital of France.'

# Good: structural
assert isinstance(result, str)
assert len(result) > 10
assert 'paris' in result.lower() or 'france' in result.lower()
```

### LLM-as-Judge

For quality (not just functional) testing, use another model call to score the output.
Especially useful when exact ground truth is not available.


In [ ]:
class MockChatModel(Runnable):
    # Drop-in replacement for any ChatModel in tests.
    # Modes:
    #   fixed response: always returns the same string
    #   callable: generate response programmatically from input
    def __init__(self, response=None, callable_=None):
        self.response   = response or 'Mock response'
        self.callable_  = callable_
        self.call_count = 0
        self.call_log   = []

    def invoke(self, input, config=None) -> AIMessage:
        self.call_count += 1
        self.call_log.append(input)
        content = self.callable_(input) if self.callable_ else self.response
        return AIMessage(content=content)

# Unit test 1: chain logic without real API
print('-- Unit Tests: Mocked model --')
prompt = ChatPromptTemplate.from_template('Translate {text} to French.')
test_chain = prompt | MockChatModel(response='Bonjour le monde') | StrOutputParser()

result = test_chain.invoke({'text': 'Hello world'})
assert isinstance(result, str),       f'Expected str, got {type(result).__name__}'
assert len(result) > 0,               'Response should not be empty'
assert result == 'Bonjour le monde',  f'Got: {result}'
print('  OK: return type is str')
print('  OK: response is non-empty')
print('  OK: mock returns deterministic output')
print()

# Unit test 2: prompt formatting (no API call needed)
print('-- Unit Tests: Prompt formatting --')
messages = prompt.invoke({'text': 'Good morning'})
msg_text  = messages.messages[0].content
assert 'Good morning' in msg_text, f'Missing input in prompt: {msg_text}'
assert 'French' in msg_text,       f'Missing instruction in prompt: {msg_text}'
print('  OK: prompt contains the input text')
print('  OK: prompt contains the instruction keyword')
print()

# Unit test 3: RunnableParallel output structure
print('-- Unit Tests: Parallel output structure --')
parallel = RunnableParallel(
    summary=MockChatModel(response='A brief summary.'),
    category=MockChatModel(response='technical'),
)
out = parallel.invoke('some text')
assert isinstance(out, dict),                         'Parallel output must be dict'
assert set(out.keys()) == {'summary', 'category'},    f'Wrong keys: {out.keys()}'
print('  OK: parallel output is a dict')
print('  OK: expected keys present')
print()

# Integration test: LLM-as-Judge
print('-- Integration Test: LLM-as-Judge --')

class EvalGrade(BaseModel):
    score:  int  = Field(description='1-10')
    passed: bool = Field(description='whether this passes the criteria')
    reason: str  = Field(description='one sentence explanation')

def evaluate(question: str, answer: str, criteria: str) -> EvalGrade:
    judge_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are an objective evaluator. Grade responses strictly.'),
        ('user',   'QUESTION: {question}\nANSWER: {answer}\nCRITERIA: {criteria}\n\nGrade 1-10.'),
    ])
    return (judge_prompt | model.with_structured_output(EvalGrade)).invoke(
        {'question': question, 'answer': answer, 'criteria': criteria}
    )

concept = 'gradient descent'
real_chain = (ChatPromptTemplate.from_template('Define {concept} in one sentence.') | model | parser)
answer = real_chain.invoke({'concept': concept})

grade = evaluate(
    question=f'Define {concept} in one sentence.',
    answer=answer,
    criteria='Accurate, under 30 words, mentions optimization.',
)
print(f'Answer: {answer}')
print(f'Score:  {grade.score}/10  Passed: {grade.passed}')
print(f'Reason: {grade.reason}')


---
## Quick Reference

```python
# 1. Structured output (no parsing needed)
chain = prompt | model.with_structured_output(MySchema)
r = chain.invoke(input)
r.field   # direct attribute access

# 2. Tool calling
@tool
def fn(arg: str) -> str:
    'Docstring = tool description'
    return result
model_wt = model.bind_tools([fn])
resp = model_wt.invoke(messages)
if resp.tool_calls:  # execute tools + loop

# 3. Custom Runnable
class MyRunnable(Runnable):
    def invoke(self, input, config=None): return transformed

# 4. Progressive enrichment
pipeline = (
    RunnablePassthrough.assign(a=chain_a)
    | RunnablePassthrough.assign(b=chain_b, c=chain_c)  # parallel
    | RunnablePassthrough.assign(final=final_chain)
)

# 5. Manual agent loop
while True:
    resp = model_wt.invoke(msgs)
    if not resp.tool_calls: return resp.content
    execute_tools(resp) ; append_ToolMessages()

# 6. Token guard
TokenGuardRunnable(chain, max_tokens=2000)

# 7. Summarization memory
memory = SummarizingMemory(max_messages=10, keep_recent=4)
memory.add(human, ai_response)
history = memory.get_history()  # -> MessagesPlaceholder

# 8. Rate limit + circuit breaker
RateLimitedRunnable(chain, rps=2.0, failure_threshold=3)

# 9. Async
result = await chain.ainvoke(input)
async for chunk in chain.astream(input): ...

# 10. Mock for tests
MockChatModel(response='Fixed string')
```

### Decision guide

| I need to... | Use |
|---|---|
| Typed output without parsing | `model.with_structured_output(Schema)` |
| Model to call a Python function | `@tool` + `model.bind_tools([...])` |
| Stateful/reusable pipeline stage | Subclass `Runnable` |
| Build context step by step | `RunnablePassthrough.assign()` chain |
| LLM agent without black-box executor | Manual `run_agent()` loop |
| Reject inputs before API call | `TokenGuardRunnable` |
| Long conversation without overflow | `SummarizingMemory` |
| Protect against rate limits + downtime | `RateLimitedRunnable` |
| Serve chains over HTTP with streaming | FastAPI + `ainvoke` / `astream` |
| Test without API calls | `MockChatModel(response='...')` |
| Score output quality automatically | `model.with_structured_output(EvalGrade)` as judge |
